# KurdiSent — Re-Annotation Evaluation

Compares the linguist's blind labels against the published gold labels.

**Inputs**
1. `labelstudio_export.csv` exported from Label Studio (must contain `id` and the choice column, default `sentiment`).
2. `annotation_key_PRIVATE.csv` the key produced by the sampler.

Analysis arms (sliced *after* labeling, the linguist saw one shuffled stream)
- Representative (~500, proportional label×category) → overall Cohen's κ, error rate.
- News (~332, representative + top-up, excl. audit-only) → κ + PABAK + raw agreement.
- Audit (~99 confident errors, conf ≥ 0.99) → reported as counts only (model-selected, biased sample).

Three labels only: Negative / Neutral / Positive. Cohen's κ is unweighted (nominal).

## 1 · Config & load

In [1]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix

EXPORT_PATH = "labelstudio_export.csv"
KEY_PATH    = "annotation_key_PRIVATE.csv"
CHOICE_COL  = "sentiment"
SEED        = 42
LABELS3     = ["Negative", "Neutral", "Positive"]
L2I         = {n: i for i, n in enumerate(LABELS3)}

exp = pd.read_csv(EXPORT_PATH, encoding="utf-8-sig")
key = pd.read_csv(KEY_PATH,   encoding="utf-8-sig")

def clean_choice(x):
    return str(x).strip().strip("[]' ").capitalize()
exp["her_label"] = exp[CHOICE_COL].map(clean_choice)

df = key.merge(exp[["id", "her_label"]], on="id", how="inner")
print(f"Joined {len(df)} of {len(key)} items.")
print("Her label distribution:", df["her_label"].value_counts().to_dict())
assert set(df["her_label"]) <= set(LABELS3), "Unexpected label values — check CHOICE_COL / cleaning."


Joined 745 of 745 items.
Her label distribution: {'Neutral': 483, 'Positive': 150, 'Negative': 112}


## 2 · Statistical helpers (Wilson CI, bootstrap κ, PABAK)

In [2]:
from math import sqrt

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = (z*sqrt(p*(1-p)/n + z*z/(4*n*n))) / d
    return p, max(0, c-h), min(1, c+h)

def kappa_boot(y_true, y_pred, B=2000, seed=SEED):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = cohen_kappa_score(y_true, y_pred)
    rng = np.random.default_rng(seed); n = len(y_true); vals = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(set(y_true[idx])) < 2 or len(set(y_pred[idx])) < 2:
            continue
        vals.append(cohen_kappa_score(y_true[idx], y_pred[idx]))
    lo, hi = (np.nanpercentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan))
    return k, lo, hi

def pabak3(po):
    return (3*po - 1) / 2

def report_block(sub, title):
    yt = sub.gold_name.map(L2I).values
    yp = sub.her_label.map(L2I).values
    po = float((yt == yp).mean()) if len(sub) else np.nan
    k, klo, khi = kappa_boot(yt, yp) if len(sub) >= 10 else (np.nan, np.nan, np.nan)
    err, elo, ehi = wilson(int((yt != yp).sum()), len(sub))
    print(f"===== {title} =====")
    print(f"  items                     : {len(sub)}")
    print(f"  raw agreement  Po         : {po:6.1%}")
    print(f"  disagreement (error) rate : {err:6.1%}  (95% CI {elo:.1%}-{ehi:.1%})")
    print(f"  Cohen's kappa (unweighted): {k:6.3f}  (95% CI {klo:.3f}-{khi:.3f})")
    print(f"  PABAK                     : {pabak3(po):6.3f}")
    cm = confusion_matrix(yt, yp, labels=[0,1,2])
    print("  confusion (rows=gold, cols=linguist)  order Neg/Neu/Pos:")
    print(pd.DataFrame(cm, index=LABELS3, columns=LABELS3).to_string().replace("\n","\n  "))
    print()
    return dict(title=title, n=len(sub), Po=po, err=err,
                kappa=k, kappa_lo=klo, kappa_hi=khi, pabak=pabak3(po))


## 3 · Overall (representative arm) & News

In [3]:
rep  = df[df.arm.str.contains("representative")]
news = df[(df.category == "news") & (df.arm.str.contains("representative|news_topup"))]

r_overall = report_block(rep,  "OVERALL  (representative, proportional label×category)")
r_news    = report_block(news, "NEWS  (representative + top-up, audit-only excluded)")


===== OVERALL  (representative, proportional label×category) =====
  items                     : 500
  raw agreement  Po         :  68.0%
  disagreement (error) rate :  32.0%  (95% CI 28.1%-36.2%)
  Cohen's kappa (unweighted):  0.520  (95% CI 0.463-0.576)
  PABAK                     :  0.520
  confusion (rows=gold, cols=linguist)  order Neg/Neu/Pos:
          Negative  Neutral  Positive
  Negative        68       92         2
  Neutral         10      147         7
  Positive         9       40       125



===== NEWS  (representative + top-up, audit-only excluded) =====
  items                     : 332
  raw agreement  Po         :  55.1%
  disagreement (error) rate :  44.9%  (95% CI 39.6%-50.3%)
  Cohen's kappa (unweighted):  0.161  (95% CI 0.100-0.228)
  PABAK                     :  0.327
  confusion (rows=gold, cols=linguist)  order Neg/Neu/Pos:
          Negative  Neutral  Positive
  Negative        11       97         0
  Neutral          2      158         1
  Positive         1       48        14



## 4 · Audit set (counts only)

In [4]:
aud = df[df.is_audit == 1].copy()

def audit_counts(sub, name):
    n = len(sub)
    sm = int((sub.her_label == sub.model_pred).sum())   # sides with model -> gold likely wrong
    sg = int((sub.her_label == sub.gold_name).sum())     # confirms gold
    ot = n - sm - sg                                     # picks the remaining class
    print(f"  {name:22s} n={n:3d} | sides_model={sm:3d}  confirms_gold={sg:3d}  other={ot:3d}")

print("===== AUDIT (confident errors, conf>=0.99) — counts only =====")
audit_counts(aud, "ALL confident errors")

flips = aud[((aud.gold_name=='Positive')&(aud.model_pred=='Negative')) |
            ((aud.gold_name=='Negative')&(aud.model_pred=='Positive'))]
neutral_boundary = aud[(aud.gold_name=='Neutral') | (aud.model_pred=='Neutral')]
audit_counts(flips,            "  of which polarity flips")
audit_counts(neutral_boundary, "  of which neutral-bdry")

print("\n  'sides_model' on the polarity-flip subset is the cleanest gold-error evidence.")
print("  Report as counts (biased, enriched sample) — do NOT convert to a dataset-wide rate.")


===== AUDIT (confident errors, conf>=0.99) — counts only =====
  ALL confident errors   n= 99 | sides_model= 23  confirms_gold= 58  other= 18
    of which polarity flips n= 34 | sides_model=  7  confirms_gold= 11  other= 16
    of which neutral-bdry n= 65 | sides_model= 16  confirms_gold= 47  other=  2

  'sides_model' on the polarity-flip subset is the cleanest gold-error evidence.
  Report as counts (biased, enriched sample) — do NOT convert to a dataset-wide rate.


## 5 · One-line summary table

In [5]:
summary = pd.DataFrame([r_overall, r_news]).set_index("title")
summary = summary[["n","Po","err","kappa","kappa_lo","kappa_hi","pabak"]]
summary.columns = ["n","Po","err_rate","kappa","k_lo","k_hi","PABAK"]
display(summary.round(3))
summary.round(4).to_csv("reannotation_summary.csv")
print("Saved reannotation_summary.csv")


,n,Po,err_rate,kappa,k_lo,k_hi,PABAK
title,,,,,,,
"OVERALL (representative, proportional label×category)",500,0.680,0.320,0.520,0.463,0.576,0.520
"NEWS (representative + top-up, audit-only excluded)",332,0.551,0.449,0.161,0.100,0.228,0.327


Saved reannotation_summary.csv
